# 📊 ACFX: Causal Feature Exploration on Health Datasets

Welcome to this Jupyter Notebook showcasing **ACFX (Automated Causal Feature Exploration)** for the purpose of a ACFX survey [LINK].
For the purpose of this survey, I prepared and attached an examplary dataset (German credit data is a dataset by Hofmann, H. (1994). Statlog (German Credit Data) UCI Machine Learning Repository. https://doi.org/10.24432/C5NC77).


## 🔍 Purpose of this Notebook

The goal of this notebook is to:
- **Load and preprocess** the datasets.
- **Split** into training and testing sets with stratification for reproducibility.
- **Explore causal relationships** among features using ACFX.
- **Visualize causal graphs** to uncover dependencies and potential actionable insights.
- Enable you to **answer questions of the survey** regarding your opinions of the framework.

By the end of this notebook, you will be able to:

- Identify **key causal relationships** in the data.  
- Understand **feature importance from a causal perspective**, not just correlation.  
- Generate **graphical visualizations** of the learned causal structures.  
- Use ACFX framework with bayesian network causability model
## Prelimiary
- Please review the prepared dataset loaded from _german_credit_numeric.csv_. Detailed descriptions of each feature are available in _legend.txt_. Feature types have been assigned according to the mapping defined in _column_types.py_.
---

In [ ]:
!python --version

In [ ]:
!pip install acfx==0.3.6

In [ ]:
!pip install ipywidgets pydot

In [ ]:
!pip install pyvis

In ACFX framework, there are three explainers available for evaluation:
1. AcfxEBM, for interpret.glassbox's ExplainableBoostingClassifier blackbox
2. AcfxLinear, for scikit-learn's LinearClassifierMixin (linear classifier with linear coeffs- LogisticRegressionCounterOptimizer is recommended here)
3. AcfxCustom, for the use of any custom blackbox (requires providing compatible counter-optimizer)

For the purpose of this survey, we will stick to the EBM blackbox and the german credit dataset

To initialize the explainer with its blackbox, run:

In [ ]:
from acfx import AcfxEBM
from interpret.glassbox import ExplainableBoostingClassifier

from column_types import feature_types

no_target_feature_types = [
    v for k, v in feature_types.items() if k != 'Credit granted'
]

categorical_indicator = [
    True if v != 'continuous' else False
    for k, v in feature_types.items()
    if k != 'Credit granted'
]

model = ExplainableBoostingClassifier(
    # Can be: 'continuous', 'nominal', or 'ordinal'
    feature_types=no_target_feature_types)
explainer = AcfxEBM(model)

In [ ]:
import pandas as pd
file_path = './german_credit_numeric.csv'

In [ ]:
bunch_df = pd.read_csv(file_path)

In [ ]:
bunch_df

In [ ]:
X = bunch_df.drop('Credit granted', axis=1)
y = bunch_df['Credit granted']

In [ ]:
# You can use seed to fix the train-test split
# SEED = 43

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.1,
# random_state=SEED
)

In [ ]:
%%capture --no-display
import logging
logging.getLogger("interpret").setLevel(logging.WARNING)
model.fit(X=X_train, y=y_train)

We will now train a pgmpy's DiscreteBayesianNetwork model that operates on discrete features. Continuous features will be discretized.

In [ ]:
%%capture --no-display
import logging
logging.getLogger("pgmpy").setLevel(logging.WARNING)

from acfx.evaluation.bayesian_model import train_bayesian_model
num_bins = 3
bayesian_model = train_bayesian_model(X_train, categorical_indicator, num_bins)


We will now display the DAG, which is generated based on the causability obtained from the Bayesian network model

In [ ]:
nodes = list(bayesian_model.nodes)
edges = list(bayesian_model.edges)

In [ ]:
import networkx as nx
G = nx.DiGraph()
G.add_nodes_from(nodes)
G.add_edges_from(edges)

In [ ]:
from pyvis.network import Network
import networkx as nx

net = Network(notebook=True, directed=True, height="600px", width="100%", bgcolor="#222222", font_color="white")

net.from_nx(G)
net.toggle_physics(True)
net.show_buttons(filter_=['physics'])
net.show("bayesian_network.html")

In [ ]:
causal_order = list(nx.topological_sort(G))
print("Causal order:", causal_order)

We will now loop through the CPDs (conditional probability distributions) to display extracted features' causability model

In [ ]:
from IPython.display import display, Markdown
limit = 3

i=0
for cpd in bayesian_model.get_cpds():
    # comment to display all
    if limit == i:
        break

    if not cpd:
        continue
    try:
        df = cpd.to_dataframe()

        display(Markdown("---"))

        display(Markdown(f"### CPD of Variable: **{cpd.variable}**"))

        evidence = cpd.get_evidence()
        if evidence:
            display(Markdown(f"*Conditioned on (Evidence):* `{', '.join(evidence)}`"))
        else:
            display(Markdown(f"*Prior Probability (Root Node - No Evidence)*"))

        display(df)
        i+=1
    except Exception as e:
        continue

In [ ]:
pbounds = {col: (X_train[col].min(), X_train[col].max()) for col in X_train.columns}
features_order = X_train.columns.tolist()

query_instance = X_test.sample(1).values

We will now fit the counterfactual explainer in order to showcase counterfactual generation process. For your convenience, counterfactual generation parameters are displayed in the cell below. You can try and manipulate them to get different results



In [ ]:

# What other features would you make not available to change?
fixed = {
    'Number of people being liable to provide maintenance for',
    'foreign worker',
    'Personal status and sex',
    "Credit amount",
    'Age'
}

PLAUSABILITY_WEIGHT=0.9
DIVERSITY_WEIGHT=0.1
SPARSITY_WEIGHT=0.1
INIT_POINTS=100

masked_features = [x for x in features_order if x not in fixed]

In [ ]:
explainer.fit(X=X_train,
              pbounds=pbounds,
              masked_features=masked_features,
              categorical_indicator=categorical_indicator,
              features_order=features_order,
              bayesian_causality=True,
              num_bins=num_bins,
              bayesian_model=bayesian_model)

In [ ]:
original_class = model.predict([query_instance])[0]
desired_class = 1 if int(original_class) == 0 else 0

In [ ]:
cf = explainer.counterfactual(desired_class=desired_class, query_instance=query_instance,plausibility_weight=PLAUSABILITY_WEIGHT, diversity_weight=DIVERSITY_WEIGHT, sparsity_weight=SPARSITY_WEIGHT,init_points=INIT_POINTS)

Counterfactual is generated. Let's see what are the proposed changes

In [ ]:
from questionnaire_utils import make_counterfactual_delta_table, counterfactual_instructions
import pandas as pd

# source code by Szymon Bobek: https://colab.research.google.com/drive/1Hj6yH4UIrAp1Jp6B1U542vcdSkXuZHUd (accessed and modified: 5 May 2026)
feature_names = X_train.columns.tolist()  # or your saved feature names
query_df = pd.DataFrame(query_instance, columns=feature_names)
cf_df = pd.DataFrame(cf, columns=feature_names)

comparison_df = pd.concat([query_df, cf_df], keys=['Original', 'Counterfactual'])

display(comparison_df)


styled, delta_df = make_counterfactual_delta_table(query_df, cf_df, decimals=3, feature_types=feature_types)

try:
    from IPython.display import display
    display(styled)
except Exception:
    print("\nRequired adjustments (delta = CF - Original):")
    print(delta_df.to_string(float_format=lambda x: f"{x:+.3f}" if abs(x) > 1e-12 else " 0.000"))

instr = counterfactual_instructions(delta_df, decimals=3, feature_types=feature_types)
print("\nInstructions:")
for k, v in instr.items():
    print(f"  {k}: {v}")
